# Phase 2: Data Understanding & Preprocessing

Objective:
- Load Zeek conn.log data
- Understand features
- Clean and preprocess data
- Prepare ML-ready dataset

In [28]:
import pandas as pd

file_path = r"C:\Users\Arushi Sharma\Downloads\Master Thesis\normal_6\normal_6\conn.log"

# Step 1: Extract column names
with open(file_path, 'r') as f:
    for line in f:
        if line.startswith('#fields'):
            columns = line.strip().split('\t')[1:]
            break

# Step 2: Load data
df = pd.read_csv(file_path, sep='\t', comment='#', names=columns)

df.head()

,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,conn_state,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents
0,1.554279e+09,CCMjy64Kn2ze7WCkUj,127.0.0.1,42100,127.0.0.1,7878,tcp,-,-,-,...,OTH,-,-,0,CcC,0,0,0,0,-
1,1.554279e+09,CVKU3t2PWKAY3uYxOd,192.168.1.190,43530,192.168.1.190,7878,tcp,-,-,-,...,OTH,-,-,0,CcC,0,0,0,0,-
2,1.554279e+09,CT5qKJgCbk0VISZK5,127.0.0.1,42100,127.0.0.1,7878,tcp,-,-,-,...,OTH,-,-,0,CcC,0,0,0,0,-
3,1.554279e+09,C9lyfla9vlc4RMewg,192.168.1.190,43530,192.168.1.190,7878,tcp,-,-,-,...,OTH,-,-,0,CcC,0,0,0,0,-
4,1.554279e+09,CRh5nYyJtzYgf4RU1,127.0.0.1,42100,127.0.0.1,7878,tcp,-,-,-,...,OTH,-,-,0,CcC,0,0,0,0,-


In [29]:
drop_cols = [
    'uid',              # random ID
    'id.orig_h',        # raw IP (high cardinality)
    'id.resp_h',        # same issue
    'local_orig',
    'local_resp',
    'tunnel_parents'
]
keep_cols = [
    'ts',
    'id.orig_p',
    'id.resp_p',
    'proto',
    'service',
    'duration',
    'orig_bytes',
    'resp_bytes',
    'conn_state',
    'missed_bytes',
    'history',
    'orig_pkts',
    'orig_ip_bytes',
    'resp_pkts',
    'resp_ip_bytes'
]
df_clean = df[keep_cols].copy()

df_clean.head()


,ts,id.orig_p,id.resp_p,proto,service,duration,orig_bytes,resp_bytes,conn_state,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes
0,1.554279e+09,42100,7878,tcp,-,-,-,-,OTH,0,CcC,0,0,0,0
1,1.554279e+09,43530,7878,tcp,-,-,-,-,OTH,0,CcC,0,0,0,0
2,1.554279e+09,42100,7878,tcp,-,-,-,-,OTH,0,CcC,0,0,0,0
3,1.554279e+09,43530,7878,tcp,-,-,-,-,OTH,0,CcC,0,0,0,0
4,1.554279e+09,42100,7878,tcp,-,-,-,-,OTH,0,CcC,0,0,0,0


In [30]:
df_clean.replace('-', 0, inplace=True)

In [31]:
numeric_cols = [
    'duration',
    'orig_bytes',
    'resp_bytes',
    'missed_bytes',
    'orig_pkts',
    'orig_ip_bytes',
    'resp_pkts',
    'resp_ip_bytes'
]

for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

In [32]:
df_clean.info()
df_clean.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11898 entries, 0 to 11897
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   ts             11898 non-null  float64
 1   id.orig_p      11898 non-null  int64  
 2   id.resp_p      11898 non-null  int64  
 3   proto          11898 non-null  object 
 4   service        11898 non-null  object 
 5   duration       11898 non-null  float64
 6   orig_bytes     11898 non-null  int64  
 7   resp_bytes     11898 non-null  int64  
 8   conn_state     11898 non-null  object 
 9   missed_bytes   11898 non-null  int64  
 10  history        11898 non-null  object 
 11  orig_pkts      11898 non-null  int64  
 12  orig_ip_bytes  11898 non-null  int64  
 13  resp_pkts      11898 non-null  int64  
 14  resp_ip_bytes  11898 non-null  int64  
dtypes: float64(2), int64(9), object(4)
memory usage: 1.4+ MB


,ts,id.orig_p,id.resp_p,duration,orig_bytes,resp_bytes,missed_bytes,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes
count,1.189800e+04,11898.000000,11898.000000,11898.000000,1.189800e+04,1.189800e+04,1.189800e+04,11898.000000,1.189800e+04,11898.000000,1.189800e+04
mean,1.554284e+09,38914.028072,8370.974954,18.756489,4.048519e+03,1.337510e+03,2.464708e+02,12.342747,2.182947e+03,10.510338,1.532978e+03
std,3.209089e+03,17403.005771,7907.136449,311.925169,4.070578e+05,9.460525e+04,1.704451e+04,613.015661,1.632563e+05,606.481896,1.152900e+05
min,1.554279e+09,3.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000e+00
25%,1.554281e+09,37371.000000,1900.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000e+00
50%,1.554284e+09,43530.000000,7878.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,1.000000,6.300000e+01,0.000000,0.000000e+00
75%,1.554287e+09,49767.750000,15600.000000,0.000054,2.400000e+01,0.000000e+00,0.000000e+00,1.000000,8.900000e+01,0.000000,0.000000e+00
max,1.554290e+09,65464.000000,52976.000000,10896.268171,4.438653e+07,9.831899e+06,1.640929e+06,52494.000000,1.748466e+07,47610.000000,1.231625e+07


In [33]:
gt = pd.read_csv(r"C:\Users\Arushi Sharma\Downloads\Master Thesis\GroundTruth_Network_6.csv")

df_clean['label'] = 0   # default = normal

gt.head()

,ts,src_ip,src_port,dst_ip,dst_port,proto,type
0,1556076944,192.168.1.79,3260,192.168.1.32,57672,tcp,scanning
1,1556076944,192.168.1.79,3260,192.168.1.32,57673,tcp,scanning
2,1556076944,192.168.1.79,32776,192.168.1.32,55204,tcp,scanning
3,1556076944,192.168.1.79,32776,192.168.1.32,57717,tcp,scanning
4,1556076944,192.168.1.79,3476,192.168.1.31,53220,tcp,scanning


In [34]:
import pandas as pd

file_path = r"C:\Users\Arushi Sharma\Downloads\Master Thesis\normal_DDoS_1\normal_DDoS_1\conn.log"

# Extract column names from Zeek header
with open(file_path, 'r') as f:
    for line in f:
        if line.startswith('#fields'):
            columns = line.strip().split('\t')[1:]
            break

# Load dataset
df_ddos = pd.read_csv(file_path, sep='\t', comment='#', names=columns)

df_ddos.head()
df_ddos.shape
df_ddos.columns


# same loading logic as before

Index(['ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p',
       'proto', 'service', 'duration', 'orig_bytes', 'resp_bytes',
       'conn_state', 'local_orig', 'local_resp', 'missed_bytes', 'history',
       'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes',
       'tunnel_parents'],
      dtype='object')

In [35]:
print(df_ddos['ts'].min(), df_ddos['ts'].max())
print(gt['ts'].min(), gt['ts'].max())

1556203726.876922 1556205014.255387
1556076944 1556085889


In [36]:
# Define attacker IP range
attacker_ips = [f"192.168.159.{i}" for i in range(30, 40)]

# Create label
df_ddos['label'] = df_ddos.apply(
    lambda row: 1 if (row['id.orig_h'] in attacker_ips or row['id.resp_h'] in attacker_ips) else 0,
    axis=1
)

In [37]:
df_model = df_ddos[[
    'ts',
    'id.orig_p',
    'id.resp_p',
    'proto',
    'service',
    'duration',
    'orig_bytes',
    'resp_bytes',
    'conn_state',
    'missed_bytes',
    'history',
    'orig_pkts',
    'orig_ip_bytes',
    'resp_pkts',
    'resp_ip_bytes',
    'label'
]].copy()

In [38]:
df_model.replace('-', 0, inplace=True)

In [39]:
numeric_cols = [
    'duration','orig_bytes','resp_bytes','missed_bytes',
    'orig_pkts','orig_ip_bytes','resp_pkts','resp_ip_bytes'
]

for col in numeric_cols:
    df_model[col] = pd.to_numeric(df_model[col], errors='coerce')

In [40]:
df_ddos['label'].value_counts()

label
0    123536
Name: count, dtype: int64

In [41]:
categorical_cols = ['proto', 'service', 'conn_state']

df_model = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)

In [42]:
df_model.dtypes

ts                   float64
id.orig_p              int64
id.resp_p              int64
duration             float64
orig_bytes             int64
resp_bytes             int64
missed_bytes           int64
history               object
orig_pkts              int64
orig_ip_bytes          int64
resp_pkts              int64
resp_ip_bytes          int64
label                  int64
proto_tcp               bool
proto_udp               bool
service_dns             bool
service_http            bool
service_ssl             bool
conn_state_REJ          bool
conn_state_RSTO         bool
conn_state_RSTOS0       bool
conn_state_RSTR         bool
conn_state_RSTRH        bool
conn_state_S0           bool
conn_state_S1           bool
conn_state_S2           bool
conn_state_S3           bool
conn_state_SF           bool
conn_state_SH           bool
conn_state_SHR          bool
dtype: object

In [43]:
df_model = df_model.apply(pd.to_numeric, errors='coerce')
df_model.fillna(0, inplace=True)

In [44]:
df_model.to_csv("df_model.csv", index=False)